In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
sys.path.append(str(Path.cwd().parent))  # project root, so "import src..." works
#Path.cwd() is notebooks so .parent is the project root.

In [ ]:
import pandas as pd
from src.data.inventory import build_reports_table, build_images_table

reports_df = build_reports_table()
images_df = build_images_table(reports_df)
images_df.head()


In [ ]:
images_df["im_part"] = images_df["image_id"].str.split("_").str[-1]
#.str is a pandas function used for columns 
images_df['im_num'] = images_df['im_part'].str.split('-').str[1]

#create 2 new columns to probably derive the patient id and see how many 
# images has that in common(im_num)


In [ ]:
images_df

In [ ]:
ims_per_report = images_df.groupby('report_id')['im_num'].nunique()
#the result of the groupby is like a small dataset

In [ ]:
ims_per_report.value_counts()
#all reports have one single image IM number except one.
# which is that report?

In [ ]:
ims_per_report[ims_per_report > 1] 


In [ ]:
images_df[images_df['report_id'] == 'CXR1015']

In [ ]:
reports_per_im = images_df.groupby('im_num')['report_id'].nunique()
reports_per_im.value_counts().sort_index()


In [ ]:
images_df[images_df['im_num'] == '0013']['report_id'].unique()
#images_df[images_df['im_num'] == '0005']['report_id'].unique()


We see consecutive files for image nmber 13 for example so that just seems to be like a counter this number which groups the files together

In [ ]:
images_df['im_int'] = images_df['im_num'].astype(int)

In [ ]:
im_by_report = images_df.groupby('report_id')['im_int'].min()

In [ ]:
im_by_report

In [ ]:
text_pos = range(len(im_by_report))
report_num = im_by_report.index.str[3:].astype(int)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(text_pos, im_by_report, s=2)
axes[0].set_title("IM number vs text-sorted position")
axes[1].scatter(report_num, im_by_report, s=2)
axes[1].set_title("IM number vs report number")


## Conclusion: is `IM-####` a patient ID?

**Test.** One IM number per report (`im_by_report`), plotted against two ways of ordering the reports:
- **Left, text-sorted position:** one straight rising line. The IM number follows the alphabetical order of the file names exactly.
- **Right, report number:** the line breaks into pieces. Text order puts `CXR101` between `CXR1009` and `CXR1010`, and puts `CXR4`, `CXR40`, `CXR400` straight after `CXR3999`. Those three even share IM 2050.

**Result.** `IM-####` is a counter from the export: it went up while going through the files in text order. It says nothing about patients. One exception: CXR1015 also has `IM-0001` images, but one report out of 3,851 doesn't change the pattern.

**Decision.** The dataset has no patient information, so each report is treated as a separate patient and the split is done **by report**.

**Limitation.** If one person has several reports, they can end up in both train and test, which would make test scores slightly too optimistic. We can't detect this, so it is documented. Grouping reports by IM number wouldn't help here: it would group neighbours in the file list, not people.